# Modelling with `dstoolkit`: from raw data to a cross-validated model

Predicting who survived the Titanic (1,309 passengers) in four steps:

1. **Explore**: `associations()` finds which columns are related to survival, and flags a column that would leak the answer.
2. **Validate**: `validate()` checks the model's inputs before any training.
3. **Model**: the cleaning runs inside a scikit-learn `Pipeline` of dstoolkit transformers, so cross-validation learns the fill values, outlier bounds and scaling from the training folds only.
4. **Read the model**: what the fitted pipeline learned, and what the coefficients say.

Needs the scikit-learn extra: `pip install "dstoolkit[sklearn]"`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline

import dstoolkit as dst

raw = pd.read_csv('../data/titanic3.csv', na_values='?')
raw.shape

(1309, 14)

## 1. What is related to survival?

`associations()` picks a suitable test for each pair of columns (Pearson or Spearman for two numbers, chi-square with Cramér's V for two categories, ANOVA with eta for one of each), corrects the p-values for testing many pairs at once (Benjamini-Hochberg), and keeps the pairs that are both significant and at least moderately strong.

The candidate features are the facts known when a passenger boarded. `on_a_lifeboat` is added on purpose: it records whether a lifeboat number appears in the `boat` column, which was only known **after** the sinking.

In [2]:
candidates = raw[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']]
pairs = dst.associations(candidates.assign(on_a_lifeboat=raw['boat'].notna()))
(pairs[['column_1', 'column_2', 'test', 'measure', 'strength', 'p_adjusted', 'n']]
 .round({'strength': 3})
 .assign(p_adjusted=lambda d: d['p_adjusted'].map(lambda p: f'{p:.0e}' if p > 0 else '< 1e-300')))

Tested 36 pair(s) of columns; 11 significant (adjusted p < 0.05, strength at least 0.3).


,column_1,column_2,test,measure,strength,p_adjusted,n
0,survived,on_a_lifeboat,ANOVA,eta,0.948,< 1e-300,1309
1,pclass,fare,Spearman,rho,-0.709,4e-199,1308
2,survived,sex,ANOVA,eta,0.529,5e-94,1309
3,sex,on_a_lifeboat,chi-square,Cramér's V,0.481,4e-67,1309
4,sibsp,fare,Spearman,rho,0.446,6e-64,1308
5,sibsp,parch,Spearman,rho,0.438,8e-62,1309
6,parch,fare,Spearman,rho,0.400,8e-51,1308
7,pclass,age,Spearman,rho,-0.396,6e-40,1046
8,pclass,embarked,ANOVA,eta,0.330,8e-33,1307
9,pclass,on_a_lifeboat,ANOVA,eta,0.323,1e-32,1309


Three things stand out:

- **`on_a_lifeboat` is almost the answer** (eta about 0.95). It's a record of the outcome, not a cause of it, so a model given it would look excellent in testing and be useless for a passenger whose fate is unknown. It stays out of the features; section 3 shows what it would do to the score.
- **Sex and class are the strongest real signals.** Women and first-class passengers survived far more often.
- **Class and fare are strongly related** (rho about -0.71: first class paid more). Both stay in, but their coefficients share the credit, so neither should be read on its own as "the effect of fare".

The other columns aren't used: `name`, `ticket` and `cabin` are identifiers or mostly empty, `body` (a body identification number) is another outcome record, and `home.dest` is free text.

## 2. Validate the inputs before training

`validate()` reports every failing row without changing the data. These rules describe what the model expects; missing values are allowed here (except for class and sex) because the pipeline fills them. In a scheduled job, `raise_if_failed()` would stop the run before a model is trained on bad data.

In [3]:
FEATURES = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X, y = raw[FEATURES], raw['survived']

report = dst.validate(X, {
    'pclass': {'type': 'integer', 'allowed': [1, 2, 3], 'required': True},
    'sex': {'allowed': ['female', 'male'], 'required': True},
    'age': {'type': 'number', 'min': 0, 'max': 100},
    'sibsp': {'type': 'integer', 'min': 0},
    'parch': {'type': 'integer', 'min': 0},
    'fare': {'type': 'number', 'min': 0},
    'embarked': {'allowed': ['C', 'Q', 'S']},
})
report.raise_if_failed()
report.summary()[['column', 'rule', 'expected', 'checked', 'failed']]

Validated 1,309 rows against 15 rule(s): all passed.


,column,rule,expected,checked,failed
0,pclass,type,integer,1309,0
1,pclass,required,no blanks,1309,0
2,pclass,allowed,"one of [1, 2, 3]",1309,0
3,sex,required,no blanks,1309,0
4,sex,allowed,"one of ['female', 'male']",1309,0
5,age,type,number,1046,0
6,age,min,≥ 0,1046,0
7,age,max,≤ 100,1046,0
8,sibsp,type,integer,1309,0
9,sibsp,min,≥ 0,1309,0


The `checked` column shows the gaps the pipeline has to handle: 263 missing ages, one missing fare and two missing ports.

(`validate()` also has named formats such as `{'format': 'us_zip'}`, `'us_phone'` and `'email'`; this dataset has no columns like that.)

## 3. A leak-free pipeline, cross-validated

Running `clean()` on all 1,309 rows before splitting would let the test rows influence the median age, the fare cap and the scaling. The transformers avoid that: inside a `Pipeline`, each cross-validation fold refits them on its training rows only.

- `MissingFiller`: median for numbers, most frequent value for text
- `OutlierCapper`: caps fares at 1.5 × IQR above the third quartile (rows are kept)
- `CategoryEncoder`: `sex` as a 0/1 code, port of embarkation as one column per port
- `Scaler`: z-scores for the numeric columns, so the coefficients are comparable

`titanic3.csv` is sorted by class, so the folds are shuffled; otherwise one fold would be almost all third class.

In [4]:
def survival_model():
    return make_pipeline(
        dst.MissingFiller(numeric='median', categorical='most_frequent'),
        dst.OutlierCapper(columns=['fare']),
        dst.CategoryEncoder({'sex': 'label', 'embarked': 'onehot'}, drop_original=True),
        dst.Scaler({c: 'zscore' for c in ['pclass', 'age', 'sibsp', 'parch', 'fare']}),
        LogisticRegression(max_iter=1000),
    )


folds = StratifiedKFold(5, shuffle=True, random_state=0)


def score(model, features):
    result = cross_validate(model, features, y, cv=folds, scoring=['accuracy', 'roc_auc'])
    return {metric: f"{result[f'test_{metric}'].mean():.3f} ± {result[f'test_{metric}'].std():.3f}"
            for metric in ['accuracy', 'roc_auc']}


pd.DataFrame({
    'always predicts "died"': score(DummyClassifier(), X),
    'logistic regression': score(survival_model(), X),
    'with on_a_lifeboat (leak)': score(survival_model(), X.assign(on_a_lifeboat=raw['boat'].notna().astype(int))),
}).T

,accuracy,roc_auc
"always predicts ""died""",0.618 ± 0.001,0.500 ± 0.000
logistic regression,0.785 ± 0.018,0.843 ± 0.010
with on_a_lifeboat (leak),0.976 ± 0.011,0.993 ± 0.004


- The **no-skill baseline** is right about 62% of the time by always predicting "died", the most common outcome. Any model has to beat that, not 50%.
- **Logistic regression** reaches about 79% accuracy, and a ROC AUC well above the baseline's 0.5, with the cleaning refitted on every fold.
- **Adding the leaked column** pushes accuracy to about 98%. That's the number to be suspicious of: it comes from information the model would never have when making a real prediction, exactly the column `associations()` flagged in step 1.

## 4. What the fitted pipeline learned

Fitted once on all the rows, the pipeline keeps everything it learned, so the same values can be applied to new passengers:

In [5]:
model = survival_model().fit(X, y)
filler, capper, encoder, scaler, logreg = model.named_steps.values()

fills = {c: filler.fill_values_[c] for c in ['age', 'fare', 'embarked']}
print('Fill values:      ', {c: v if isinstance(v, str) else float(v) for c, v in fills.items()})
print('Fare capped above:', round(capper.bounds_['fare'][1], 2))
print('Sex codes:        ', dict(enumerate(encoder.categories_['sex'])))

Fill values:       {'age': 28.0, 'fare': 14.4542, 'embarked': 'S'}
Fare capped above: 66.34
Sex codes:         {0: 'female', 1: 'male'}


The coefficients, as odds ratios: above 1 raises the odds of survival, below 1 lowers them. Numeric columns are per standard deviation; `sex_label` and the port columns compare 1 with 0.

In [6]:
odds = pd.Series(np.exp(logreg.coef_[0]), index=model[:-1].get_feature_names_out())
odds.sort_values().round(2).to_frame('odds ratio')

,odds ratio
sex_label,0.08
pclass,0.50
age,0.63
sibsp,0.70
embarked_S,0.81
embarked_Q,0.87
parch,0.96
fare,1.23
embarked_C,1.41


Being male (`sex_label` = 1) cuts the odds of survival most, and each step toward third class lowers them further, matching what `associations()` found before any model was trained. Older passengers and those travelling with several siblings or a spouse also fared worse.

These are associations in a model, not causes: class and fare overlap (step 1), so their separate odds ratios depend on each other.

## Where to go next

- [`Profile_Titanic`](Profile_Titanic.ipynb): profile the raw data and apply the suggested cleaning code
- [`Preprocessing_Titanic`](Preprocessing_Titanic.ipynb): each cleaning step on its own, for exploration rather than modelling